# transwarp-conduit 00 · From target weights to orders, rehearsed

Execution through Interactive Brokers: target weights become a checked **order sheet**, the sheet
becomes market-on-close orders, fills come back and are reconciled against the decision price.
This notebook rehearses the whole path against the package's **simulated broker** (`SimBroker`),
with real prices and volumes from Massive through unimatrix-zero's return panel. **Nothing here
reaches Interactive Brokers**: the one cell that touches a real Gateway only reads the account,
and skips when no Gateway is running.

1. settings and limits
2. a simulated broker, and matching names
3. contract mapping
4. planning a sheet: the checks
5. submitting to the simulator, fills, reconciliation
6. the next rebalance: holdings, positions that changed, the cutoff
7. a read-only look at the real paper account

In [1]:
import tempfile
from datetime import UTC, date, datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import polars as pl
from massive_subspace_stream import Store
from subspace_probe import style
from transwarp_conduit import (
    Conduit,
    ConduitError,
    ConduitSettings,
    Limits,
    SimBroker,
    ib_symbol,
    is_paper,
    load_settings,
    names_match,
)
from unimatrix_zero import Engine

style()
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_width_chars(220)
pl.Config.set_tbl_cols(24)
store = Store()
engine = Engine(store=store)

## 1 · Settings and limits

Settings come from `[transwarp_conduit]` in `collective.toml` and `TRANSWARP_*` variables. There
are no credentials: IB Gateway holds the login. Limits are checked on every sheet; a failed check
blocks it.

In [2]:
real = load_settings()
print({k: v for k, v in real.__dict__.items() if k not in ("limits", "account")} | {"account": "set" if real.account else None})
pl.DataFrame({"limit": list(Limits().__dict__), "default": list(Limits().__dict__.values())})

{'host': '127.0.0.1', 'port': 4002, 'client_id': 17, 'live': False, 'capital': None, 'order_type': 'MOC', 'cutoff_minutes': 15, 'contract_max_age_days': 30, 'timeout': 10.0, 'root': None, 'account': None}


limit,default
str,f64
"""max_weight""",0.03
"""max_gross""",2.1
"""max_net""",0.1
"""max_turnover""",2.5
"""max_order_usd""",250000.0
"""max_participation""",0.1
"""min_trade_usd""",200.0
"""min_trade_weight""",0.0005


## 2 · A simulated broker, and matching names

`SimBroker` answers the same calls as the IB adapter (account, positions, contract details, FIGI
lookups, orders, fills) in memory. Its contract list is built here from the security master for a
dozen large companies, the way IB would describe them. Names matter: when IB does not know a
FIGI, a symbol lookup is accepted only if the company name matches after legal forms and security
descriptions are dropped. `names_match` on real pairs:

In [3]:
pairs = [("McDonald's Corporation", "MC DONALD'S-CORP"), ("Mercado Libre, Inc", "MERCADOLIBRE INC"),
         ("Charter Comm Inc Del CL A New", "CHARTER COMMUNICATIONS INC-A"),
         ("Alibaba Group Holding Limited American Depositary Shares", "ALIBABA GROUP HOLDING-SP ADR"),
         ("U-Haul Holding Company", "U-HAUL HOLDING CO-NON VOTING"),
         ("First Solar, Inc.", "FIRST HORIZON CORP"), ("Apple Inc.", "APPLE HOSPITALITY REIT INC")]
pl.DataFrame({"massive_name": [a for a, _ in pairs], "ib_name": [b for _, b in pairs], "match": [names_match(a, b) for a, b in pairs]})

massive_name,ib_name,match
str,str,bool
"""McDonald's Corporation""","""MC DONALD'S-CORP""",true
"""Mercado Libre, Inc""","""MERCADOLIBRE INC""",true
"""Charter Comm Inc Del CL A New""","""CHARTER COMMUNICATIONS INC-A""",true
"""Alibaba Group Holding Limited …","""ALIBABA GROUP HOLDING-SP ADR""",true
"""U-Haul Holding Company""","""U-HAUL HOLDING CO-NON VOTING""",true
"""First Solar, Inc.""","""FIRST HORIZON CORP""",false
"""Apple Inc.""","""APPLE HOSPITALITY REIT INC""",false


In [4]:
TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "BRK.B", "JPM", "XOM", "JNJ", "V", "PG"]
ON = date(2026, 11, 3)                                   # the last close in the data when this was written
ids = store.resolve(TICKERS, on=ON).select("ticker", "security_id", "name")
master = store.security_master(TICKERS).filter(pl.col("valid_to").is_null()).select("ticker", "security_id", "primary_exchange", "composite_figi")
names = ids.join(master, on=["ticker", "security_id"])
contracts, figis = {}, {}
for i, r in enumerate(names.iter_rows(named=True), start=1001):
    sym = ib_symbol(r["ticker"])
    contracts[sym] = [{"con_id": i, "primary_exchange": {"XNAS": "NASDAQ", "XNYS": "NYSE"}.get(r["primary_exchange"], "NYSE"),
                       "long_name": r["name"].upper()}]
    figis[r["composite_figi"]] = sym
sim = SimBroker(contracts=contracts, figis=figis, cash=1_000_000.0)
print(f"{len(contracts)} contracts; account {sim.account_id()} (paper: {is_paper(sim.account_id())})")
names.select("ticker", "security_id", "name", "composite_figi")

12 contracts; account DU0000001 (paper: True)


ticker,security_id,name,composite_figi
str,str,str,str
"""AAPL""","""BBG000B9XRY4""","""Apple Inc.""","""BBG000B9XRY4"""
"""AMZN""","""BBG000BVPV84""","""Amazon.Com Inc""","""BBG000BVPV84"""
"""BRK.B""","""BBG000DWG505""","""BERKSHIRE HATHAWAY Class B""","""BBG000DWG505"""
"""GOOGL""","""BBG009S39JX6""","""Alphabet Inc. Class A Common S…","""BBG009S39JX6"""
"""JNJ""","""BBG000BMHYD1""","""Johnson & Johnson""","""BBG000BMHYD1"""
"""JPM""","""BBG000DMBXR2""","""JPMorgan Chase & Co.""","""BBG000DMBXR2"""
"""META""","""BBG000MM2P62""","""Meta Platforms, Inc. Class A C…","""BBG000MM2P62"""
"""MSFT""","""BBG000BPH459""","""Microsoft Corp""","""BBG000BPH459"""
"""NVDA""","""BBG000BBJQV0""","""Nvidia Corp""","""BBG000BBJQV0"""


## 3 · Contract mapping

`ContractBook.resolve` tries the listing's FIGI first (an exact identity), then the symbol with
the name and exchange checks. Checked matches are cached under the data root for 30 days. The
journal and cache of this rehearsal live in a temporary folder, so the real journal is untouched.

In [5]:
root = Path(tempfile.mkdtemp(prefix="conduit-demo-"))
CLOCK = datetime(2026, 11, 4, 15, 0, tzinfo=UTC)         # 10:00 ET on a trading day: before the MOC cutoff
settings = ConduitSettings(root=str(root), capital=1_000_000.0)
c = Conduit(settings, broker=sim, store=store, engine=engine, clock=lambda: CLOCK)
book = c.book.resolve(names["security_id"].to_list(), ON, sim)
book.select("security_id", "ticker", "ib_symbol", "con_id", "ib_exchange", "ib_name", "method", "status", "note")

security_id,ticker,ib_symbol,con_id,ib_exchange,ib_name,method,status,note
str,str,str,i64,str,str,str,str,str
"""BBG000B9XRY4""","""AAPL""","""AAPL""",1001,"""NASDAQ""","""APPLE INC.""","""figi""","""ok""",null
"""BBG000BBJQV0""","""NVDA""","""NVDA""",1009,"""NASDAQ""","""NVIDIA CORP""","""figi""","""ok""",null
"""BBG000BMHYD1""","""JNJ""","""JNJ""",1005,"""NYSE""","""JOHNSON & JOHNSON""","""figi""","""ok""",null
"""BBG000BPH459""","""MSFT""","""MSFT""",1008,"""NASDAQ""","""MICROSOFT CORP""","""figi""","""ok""",null
"""BBG000BR2TH3""","""PG""","""PG""",1010,"""NYSE""","""PROCTER & GAMBLE COMPANY""","""figi""","""ok""",null
"""BBG000BVPV84""","""AMZN""","""AMZN""",1002,"""NASDAQ""","""AMAZON.COM INC""","""figi""","""ok""",null
"""BBG000DMBXR2""","""JPM""","""JPM""",1006,"""NYSE""","""JPMORGAN CHASE & CO.""","""figi""","""ok""",null
"""BBG000DWG505""","""BRK.B""","""BRK B""",1003,"""NYSE""","""BERKSHIRE HATHAWAY CLASS B""","""figi""","""ok""",null
"""BBG000MM2P62""","""META""","""META""",1007,"""NASDAQ""","""META PLATFORMS, INC. CLASS A C…","""figi""","""ok""",null


## 4 · Planning a sheet: the checks

Targets are weights of capital. Shares are whole (weight × capital ÷ last close), trades below
the minimum size are skipped unless they close a position, and each order shows the cost model's
estimate. Per-order checks: a tradable contract, a price, order size, participation in average
dollar volume, borrow availability for shorts (unknown here: the simulator has no borrow data, so
it is a warning). Per-sheet checks: weight per name, gross, net and turnover.

In [6]:
WEIGHTS = {"AAPL": 0.028, "MSFT": 0.028, "NVDA": 0.025, "AMZN": 0.025, "META": 0.02, "GOOGL": 0.02, "BRK.B": 0.015,
           "JPM": 0.015, "XOM": -0.025, "JNJ": -0.025, "V": -0.02, "PG": -0.02}       # within the default limits
targets = names.select("security_id", pl.col("ticker").replace_strict(WEIGHTS, return_dtype=pl.Float64).alias("weight"))
sheet = c.plan(targets)
print(sheet.summary())
sheet.orders.select("ticker", "price", "target_weight", "current_shares", "target_shares", "trade_shares", "action",
                    "trade_value", "est_cost_bps", "est_cost_usd", "shortable", "status", "note")

{'sheet_id': '20261104-c57aa5', 'status': 'planned', 'price_date': '2026-11-03', 'capital': 1000000.0, 'orders': 12, 'buy_usd': 175758.65, 'sell_usd': 90155.38, 'turnover': 0.2659, 'gross_after': 0.2659, 'net_after': 0.0856, 'est_cost_usd': 50.24, 'skipped_small': 0, 'blocked': 0, 'unmanaged_positions': 0, 'issues': [], 'warnings': ['4 short(s) with unknown borrow availability']}


ticker,price,target_weight,current_shares,target_shares,trade_shares,action,trade_value,est_cost_bps,est_cost_usd,shortable,status,note
str,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,str,str
"""MSFT""",520.99,0.028,0.0,54.0,54.0,"""BUY""",28133.46,1.720852,4.841352,null,"""send""",null
"""AAPL""",331.9,0.028,0.0,84.0,84.0,"""BUY""",27879.6,1.667777,4.649696,null,"""send""",null
"""AMZN""",250.69,0.025,0.0,100.0,100.0,"""BUY""",25069.0,1.773723,4.446546,null,"""send""",null
"""XOM""",177.46,-0.025,0.0,-141.0,-141.0,"""SELL""",-25021.86,2.194168,5.490216,null,"""send""","""borrow unknown"""
"""JNJ""",266.17,-0.025,0.0,-94.0,-94.0,"""SELL""",-25019.98,2.202723,5.511209,null,"""send""","""borrow unknown"""
"""NVDA""",206.69,0.025,0.0,121.0,121.0,"""BUY""",25009.49,1.735356,4.340037,null,"""send""",null
"""V""",420.42,-0.02,0.0,-48.0,-48.0,"""SELL""",-20180.16,1.945546,3.926143,null,"""send""","""borrow unknown"""
"""PG""",162.06,-0.02,0.0,-123.0,-123.0,"""SELL""",-19933.38,2.175661,4.336828,null,"""send""","""borrow unknown"""
"""GOOGL""",280.03,0.02,0.0,71.0,71.0,"""BUY""",19882.13,1.785649,3.550251,null,"""send""",null


A sheet that breaks a limit is `blocked` and cannot be submitted; the issues say why. The same
targets against a 2% per-name limit and a $20,000 per-order limit (in a folder of its own: with a
fixed clock and identical targets the sheet id would otherwise repeat):

In [7]:
tight = Conduit(ConduitSettings(root=str(root / "tight"), capital=1_000_000.0, limits=Limits(max_weight=0.02, max_order_usd=20_000)),
                broker=sim, store=store, engine=engine, clock=lambda: CLOCK)
blocked = tight.plan(targets)
print(blocked.status)
for issue in blocked.issues:
    print("-", issue[:200])
try:
    tight.submit(blocked.sheet_id)
except ConduitError as exc:
    print("submit refused:", str(exc)[:120])

blocked
- 7 order(s) blocked: AAPL: order size; XOM: order size; JNJ: order size; AMZN: order size; MSFT: order size; NVDA: order size; V: order size
submit refused: sheet 20261104-c57aa5 is blocked: 7 order(s) blocked: AAPL: order size; XOM: order size; JNJ: order size; AMZN: order si


## 5 · Submitting to the simulator, fills, reconciliation

`submit` re-checks everything (planned today, before the cutoff, same account, positions
unchanged, no open orders from an earlier sheet, paper account) and only then sends. The broker
here is the simulator: `fill()` plays the closing auction at prices 20 basis points worse than the
decision price, and `reconcile` compares expected and actual positions and measures slippage.

In [8]:
placed = c.submit(sheet.sheet_id)
print(placed.select("order_id", "symbol", "action", "quantity", "order_type", "order_ref", "status"))
decision = {int(r["con_id"]): r["price"] for r in sheet.to_send.iter_rows(named=True)}
worse = {cid: px * (1.002 if r["action"] == "BUY" else 0.998) for (cid, px), r in zip(decision.items(), sheet.to_send.iter_rows(named=True), strict=True)}
sim.fill(worse)
rec = c.reconcile(sheet.sheet_id)
rec.select("ticker", "current_shares", "target_shares", "ordered", "filled", "expected", "actual", "diff", "unfilled",
           "decision_price", "avg_fill_price", "slippage_bps", "commission")

shape: (12, 7)
┌──────────┬────────┬────────┬──────────┬────────────┬─────────────────┬───────────┐
│ order_id ┆ symbol ┆ action ┆ quantity ┆ order_type ┆ order_ref       ┆ status    │
│ ---      ┆ ---    ┆ ---    ┆ ---      ┆ ---        ┆ ---             ┆ ---       │
│ i64      ┆ str    ┆ str    ┆ f64      ┆ str        ┆ str             ┆ str       │
╞══════════╪════════╪════════╪══════════╪════════════╪═════════════════╪═══════════╡
│ 1        ┆ MSFT   ┆ BUY    ┆ 54.0     ┆ MOC        ┆ 20261104-c57aa5 ┆ Submitted │
│ 2        ┆ AAPL   ┆ BUY    ┆ 84.0     ┆ MOC        ┆ 20261104-c57aa5 ┆ Submitted │
│ 3        ┆ AMZN   ┆ BUY    ┆ 100.0    ┆ MOC        ┆ 20261104-c57aa5 ┆ Submitted │
│ 4        ┆ XOM    ┆ SELL   ┆ 141.0    ┆ MOC        ┆ 20261104-c57aa5 ┆ Submitted │
│ 5        ┆ JNJ    ┆ SELL   ┆ 94.0     ┆ MOC        ┆ 20261104-c57aa5 ┆ Submitted │
│ 6        ┆ NVDA   ┆ BUY    ┆ 121.0    ┆ MOC        ┆ 20261104-c57aa5 ┆ Submitted │
│ 7        ┆ V      ┆ SELL   ┆ 48.0     ┆ MOC     

ticker,current_shares,target_shares,ordered,filled,expected,actual,diff,unfilled,decision_price,avg_fill_price,slippage_bps,commission
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""MSFT""",0.0,54.0,54.0,54.0,54.0,54.0,0.0,0.0,520.99,522.03198,20.0,0.189
"""AAPL""",0.0,84.0,84.0,84.0,84.0,84.0,0.0,0.0,331.9,332.5638,20.0,0.294
"""AMZN""",0.0,100.0,100.0,100.0,100.0,100.0,0.0,0.0,250.69,251.19138,20.0,0.35
"""XOM""",0.0,-141.0,-141.0,-141.0,-141.0,-141.0,0.0,0.0,177.46,177.10508,20.0,0.4935
"""JNJ""",0.0,-94.0,-94.0,-94.0,-94.0,-94.0,0.0,0.0,266.17,265.63766,20.0,0.329
"""NVDA""",0.0,121.0,121.0,121.0,121.0,121.0,0.0,0.0,206.69,207.10338,20.0,0.4235
"""V""",0.0,-48.0,-48.0,-48.0,-48.0,-48.0,0.0,0.0,420.42,419.57916,20.0,0.168
"""PG""",0.0,-123.0,-123.0,-123.0,-123.0,-123.0,0.0,0.0,162.06,161.73588,20.0,0.4305
"""GOOGL""",0.0,71.0,71.0,71.0,71.0,71.0,0.0,0.0,280.03,280.59006,20.0,0.2485


In [9]:
print([e["event"] for e in c.journal.events()])
print(f"fills journaled: {c.journal.fills().height}; sheet status: {c.sheet(sheet.sheet_id).status}")

['plan', 'submitting', 'submit', 'fills', 'reconcile']
fills journaled: 12; sheet status: submitted


## 6 · The next rebalance

`holdings()` returns the managed positions as weights of capital, the starting point for the next
optimization. A new target set that drops two names closes them; positions this package never ordered
are left alone and reported. And if positions change between planning and submitting, the sheet
is refused: plan again.

In [10]:
print(c.holdings().join(names.select("security_id", "ticker"), on="security_id").select("ticker", "weight").sort("weight", descending=True))
dropped = names.filter(pl.col("ticker").is_in(["BRK.B", "JPM"]))["security_id"].to_list()
next_targets = targets.filter(~pl.col("security_id").is_in(dropped))
sheet2 = c.plan(next_targets)
print(sheet2.summary()["orders"], "orders;", "closing:", sheet2.to_send.filter(pl.col("target_shares") == 0)["ib_symbol"].to_list())
sim.holdings[1001] += 10                                  # something else traded in the account meanwhile
try:
    c.submit(sheet2.sheet_id)
except ConduitError as exc:
    print("submit refused:", exc)

shape: (12, 2)
┌────────┬───────────┐
│ ticker ┆ weight    │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ MSFT   ┆ 0.028133  │
│ AAPL   ┆ 0.02788   │
│ AMZN   ┆ 0.025069  │
│ NVDA   ┆ 0.025009  │
│ GOOGL  ┆ 0.019882  │
│ META   ┆ 0.019873  │
│ BRK.B  ┆ 0.015041  │
│ JPM    ┆ 0.014871  │
│ PG     ┆ -0.019933 │
│ V      ┆ -0.02018  │
│ JNJ    ┆ -0.02502  │
│ XOM    ┆ -0.025022 │
└────────┴───────────┘
2 orders; closing: ['BRK B', 'JPM']
submit refused: positions changed since the sheet was planned; plan again


In [11]:
for day in (date(2026, 11, 4), date(2026, 11, 7), date(2026, 11, 26)):   # a trading day, a Saturday, Thanksgiving
    cut = c.cutoff(day)
    print(day, "->", cut.astimezone(ZoneInfo("America/New_York")).strftime("%H:%M %Z") if cut else "not a trading day")

2026-11-04 -> 15:45 EST
2026-11-07 -> not a trading day
2026-11-26 -> not a trading day


## 7 · A read-only look at the real paper account

`Conduit()` with the real settings opens a read-only connection to IB Gateway. `status` reads the
account (masked), positions and open orders; it sends nothing. When no Gateway is running the
cell says so and moves on.

In [12]:
try:
    print(Conduit().status())
except ConduitError as exc:
    print("no Gateway:", str(exc)[:160])

API connection failed: ConnectionRefusedError(61, "Connect call failed ('127.0.0.1', 4002)")


Make sure API port on TWS/IBG is open


no Gateway: cannot connect to IB at 127.0.0.1:4002 (ConnectionRefusedError: [Errno 61] Connect call failed ('127.0.0.1', 4002)). Is IB Gateway or TWS running and logged in,


## Takeaways
- Weights in, a saved and checked order sheet out; `submit` is the only place anything is sent,
  and only after every check passes again.
- Contracts are matched by FIGI first, then by symbol with a name check; everything is cached
  and overridable by hand.
- The simulator exercises the same path as IB, so a rebalance can be rehearsed end to end
  before the Gateway is involved.
- Real orders are paper-first: a live account needs an explicit flag, a setting and the account
  id to agree. Never submit from a notebook without the owner's go-ahead for that sheet.